# NeuroGraph · Phase 7 · QLoRA fine-tune of the university assistant
Settings: Accelerator **GPU T4 x2**, Internet **On**. Input: the **neurograph-60-intake** notebook output (redacted university chunks). Secrets (**Add-ons → Secrets**, attached): `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`.
Also runs locally (a checkout with its `.venv` and a gitignored `.env`): the setup cells detect Kaggle and otherwise skip the clone, the installs and the Secrets. Locally the notebook 60 output is found under `NEUROGRAPH_INPUT_ROOT` (stands in for `/kaggle/input`).

## 1. Protect preinstalled packages

In [ ]:
from importlib.metadata import version

def write_constraints(pkgs: list[str], path: str) -> None:
    with open(path, "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in pkgs))

write_constraints(["torch", "numpy", "scipy", "scikit-learn", "mne"], "/tmp/keep.txt")
print(open("/tmp/keep.txt").read())

## 2. Get the latest code + install (adds peft + bitsandbytes for 4-bit training) on Kaggle, or use this checkout (local)

In [ ]:
import os, sys
KAGGLE = os.path.isdir("/kaggle/working")
if KAGGLE:
    R, PY = "/kaggle/working/neurograph", "python"
    if not os.path.isdir(R):
        !git clone -q https://github.com/stuBBornButterfly/neurograph.git {R}
    !git -C {R} pull -q
else:  # local run: this checkout and its .venv (packages already installed)
    R, PY = os.environ.get("NEUROGRAPH_REPO") or os.path.abspath(".."), sys.executable
!git -C {R} log --oneline -1
%cd {R}
if KAGGLE:
    !pip install -q -r requirements.txt -c /tmp/keep.txt
    !pip install -q -r requirements-finetune.txt -c /tmp/keep.txt
    !pip install -q -e . -c /tmp/keep.txt

## 3. AuraDB credentials from Kaggle Secrets or `.env` (never printed) + notebook 60 output into the working folder

In [ ]:
import glob, shutil
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
        os.environ[key] = secrets.get_secret(key)
    D, inp = "/kaggle/working/derivatives", "/kaggle/input"
else:  # local run: the gitignored .env (Neo4j login + NEUROGRAPH_* folders; variables already set win)
    from dotenv import load_dotenv
    load_dotenv(f"{R}/.env")
    D, inp = os.environ["NEUROGRAPH_DERIVATIVES_DIR"], os.environ["NEUROGRAPH_INPUT_ROOT"]
src = os.path.dirname(glob.glob(f"{inp}/**/institutional/chunks.jsonl", recursive=True)[0])
if os.path.realpath(src) != os.path.realpath(f"{D}/institutional"):
    shutil.copytree(src, f"{D}/institutional", dirs_exist_ok=True)
print("loaded:", [k for k in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"] if os.environ.get(k)], "| chunks from", src)

## 4. Step 7.1: Q&A pairs from the university chunks (Qwen2.5-7B teacher, NLI filter, held-out chunks)

In [ ]:
!cd {R} && {PY} -m scripts.make_qa_dataset 2>&1 | grep -v -i -e warn -e it/s -e Loading

## 5. Step 7.2: QLoRA training (4-bit Qwen2.5-3B + LoRA adapters)

In [ ]:
!cd {R} && {PY} -m scripts.train_qlora 2>&1 | grep -v -i -e warn -e it/s -e Loading

## 6. Step 7.3: base vs fine-tuned on held-out chunks + model card

In [ ]:
!cd {R} && {PY} -m scripts.eval_qlora 2>&1 | grep -v -i -e warn -e it/s -e Loading

## 7. The whole agent with the fine-tuned model (same 24 questions as notebook 60): adapter on every route, then on the policy route only (the one it was trained on)

In [ ]:
!cd {R} && {PY} -m scripts.run_agent_eval --adapter {D}/qlora/adapter --tag _qlora 2>&1 | grep -v -i -e warn -e it/s -e Loading
!cd {R} && {PY} -m scripts.run_agent_eval --adapter {D}/qlora/adapter --adapter-policy-only --tag _qlora_policy 2>&1 | grep -v -i -e warn -e it/s -e Loading

## 8. Step 7.4: merge the adapter into the base + Q4_K_M GGUF for the CPU demo (llama.cpp)
Optional upload to a Hugging Face model repo: add the secrets `HF_TOKEN` (write token) and `HF_REPO` (e.g. `<user>/neurograph-qwen2.5-3b-gguf`).

In [ ]:
if KAGGLE:  # optional; without them the GGUF stays in the notebook output (local: HF_TOKEN / HF_REPO from .env, if set)
    for key in ["HF_TOKEN", "HF_REPO"]:
        try:
            os.environ[key] = secrets.get_secret(key)
        except Exception:
            pass
!cd {R} && {PY} -m scripts.export_gguf 2>&1 | grep -v -i -e warn -e it/s -e Loading